## BGE Encoder test

In [23]:
import importlib,json,numpy as np,pandas as pd
from sentence_transformers import SentenceTransformer
import src.paths as paths,src.defects as defects,src.representations as representations
importlib.reload(paths);importlib.reload(defects);importlib.reload(representations)

from src.paths import get_project_paths
from src.defects import make_missing_cross_channel_plan,inject_missing_cross_channel_reference
from src.representations import serialize_campaign_dataset,encode_serialized_campaigns,split_campaign_into_chunks

PATHS=get_project_paths()
DATA_DIR=PATHS["processed"];RESULTS_DIR=PATHS["results"]
SEVERITIES=[.10,.30,.60]

In [24]:
# load clean data
clean_campaigns=pd.read_csv(DATA_DIR/"clean_structured_campaigns.csv",parse_dates=["timestamp"])
split_df=pd.read_csv(DATA_DIR/"structural_campaign_split.csv")
eval_ids=set(split_df.loc[split_df["split"]=="evaluation","campaign_id"])
clean_eval=clean_campaigns[clean_campaigns["campaign_id"].isin(eval_ids)].copy().reset_index(drop=True)

assert clean_eval["campaign_id"].nunique()==300
print("Evaluation campaigns:",clean_eval["campaign_id"].nunique())
print("Messages:",len(clean_eval))

Evaluation campaigns: 300
Messages: 2090


In [25]:
# recover defect plan
missing_plan=make_missing_cross_channel_plan(clean_eval,seed=42)

print("Plan rows:",len(missing_plan))
print("Unique campaigns:",missing_plan["campaign_id"].nunique())
display(missing_plan.head())

assert len(missing_plan)==300
assert missing_plan["campaign_id"].nunique()==300

Plan rows: 300
Unique campaigns: 300


,campaign_id,target_message_id,target_message_order,reference_id,reference_message_order,target_channel,original_reference_channel,replacement_channel,sender,original_text,severity_rank
0,C0003,C0003_M03,3,C0003_M01,1,email,sms,whatsapp,advisor_07,Following up here on the sms message I sent ea...,27
1,C0004,C0004_M03,3,C0004_M01,1,sms,email,whatsapp,advisor_17,Following up here on the email message I sent ...,256
2,C0006,C0006_M03,3,C0006_M01,1,whatsapp,sms,email,advisor_06,Following up here on the sms message I sent ea...,46
3,C0008,C0008_M03,3,C0008_M01,1,sms,email,whatsapp,advisor_15,Following up here on the email message I sent ...,41
4,C0010,C0010_M03,3,C0010_M01,1,sms,email,whatsapp,advisor_17,Following up here on the email message I sent ...,141


In [26]:
# reconstruct conditions
missing_conditions={"clean":clean_eval.copy()}
missing_audits={}

for s in SEVERITIES:
    defective,audit=inject_missing_cross_channel_reference(clean_eval,missing_plan,severity=s)
    missing_conditions[s]=defective
    missing_audits[s]=audit
    print(f"{s:.0%} | affected =",len(audit))

10% | affected = 30
30% | affected = 90
60% | affected = 180


In [27]:
# validate nesting
affected={s:set(missing_audits[s]["campaign_id"]) for s in SEVERITIES}

assert affected[.10].issubset(affected[.30])
assert affected[.30].issubset(affected[.60])
assert [len(affected[s]) for s in SEVERITIES]==[30,90,180]

print("Nested affected sets passed.")

Nested affected sets passed.


In [28]:
# compare frozen affected IDs
saved_affected=pd.read_csv(DATA_DIR/"missing_reference_affected_campaign_ids.csv")

for s in SEVERITIES:
    reconstructed=set(missing_audits[s]["campaign_id"].astype(str))
    frozen=set(saved_affected.loc[np.isclose(saved_affected["severity"],s),"campaign_id"].astype(str))
    print(f"{s:.0%} | reconstructed={len(reconstructed)} | frozen={len(frozen)} | exact match={reconstructed==frozen}")
    assert reconstructed==frozen

10% | reconstructed=30 | frozen=30 | exact match=True
30% | reconstructed=90 | frozen=90 | exact match=True
60% | reconstructed=180 | frozen=180 | exact match=True


In [29]:
# verify only text changed
NON_TEXT=["campaign_id","message_id","message_order","timestamp","channel","sender","recipient","entity_id","reference_id","parent_message_id"]

for s in SEVERITIES:
    defective=missing_conditions[s]
    assert clean_eval[NON_TEXT].reset_index(drop=True).equals(defective[NON_TEXT].reset_index(drop=True))

    clean_i=clean_eval.set_index("message_id")
    defective_i=defective.set_index("message_id")
    changed=(clean_i["text"]!=defective_i["text"])
    changed_messages=changed[changed].index
    changed_campaigns=clean_i.loc[changed_messages,"campaign_id"].value_counts()

    assert (changed_campaigns==1).all()
    assert len(changed_messages)==int(s*300)
    print(f"{s:.0%} | changed messages={len(changed_messages)}")

print("Non-text preservation passed.")

10% | changed messages=30
30% | changed messages=90
60% | changed messages=180
Non-text preservation passed.


In [30]:
# serialize
missing_serialized={k:serialize_campaign_dataset(df) for k,df in missing_conditions.items()}

for k,df in missing_serialized.items():
    print(k,df.shape)
    assert df.shape==(300,2)

clean (300, 2)
0.1 (300, 2)
0.3 (300, 2)
0.6 (300, 2)


In [31]:
# load frozen MiniLM
with open(RESULTS_DIR/"semantic_embedding_configuration.json") as f:
    semantic_config=json.load(f)

MINILM_MODEL_NAME=semantic_config["model"]
MINILM_REVISION=semantic_config["revision"]

minilm_model=SentenceTransformer(MINILM_MODEL_NAME,revision=MINILM_REVISION)

print("Model:",MINILM_MODEL_NAME)
print("Revision:",MINILM_REVISION)
print("Dimension:",minilm_model.get_sentence_embedding_dimension())
print("Max sequence length:",minilm_model.max_seq_length)

Model: sentence-transformers/all-MiniLM-L6-v2
Revision: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
Dimension: 384
Max sequence length: 256


In [32]:
# reconstruct MiniLM embeddings
reconstructed_minilm={}

for c in ["clean",.10,.30,.60]:
    X,runtime=encode_serialized_campaigns(missing_serialized[c],minilm_model,batch_size=64)
    reconstructed_minilm[c]=X
    print(c,X.shape,f"{runtime:.2f}s")

Token indices sequence length is longer than the specified maximum sequence length for this model (259 > 256). Running this sequence through the model will result in indexing errors


Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

clean (300, 384) 40.60s
Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

0.1 (300, 384) 39.01s
Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

0.3 (300, 384) 41.71s
Campaigns: 300
Chunks to encode: 309


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

0.6 (300, 384) 36.96s


In [33]:
mini_chunk_check=[]

for c in ["clean",.10,.30,.60]:
    for _,row in missing_serialized[c].iterrows():
        chunks=split_campaign_into_chunks(row["serialized_text"],minilm_model)
        lengths=[len(minilm_model.tokenizer(x,add_special_tokens=True,truncation=False)["input_ids"]) for x in chunks]
        mini_chunk_check.append({"condition":c,"n_chunks":len(chunks),"max_tokens":max(lengths)})

mini_chunk_check=pd.DataFrame(mini_chunk_check)

display(mini_chunk_check.groupby("condition").agg(
    campaigns=("n_chunks","size"),
    multi_chunk=("n_chunks",lambda x:(x>1).sum()),
    max_chunks=("n_chunks","max"),
    max_tokens=("max_tokens","max")
).reset_index())

assert (mini_chunk_check["max_tokens"]<=minilm_model.max_seq_length).all()
print("All actual MiniLM chunks are within the model limit.")

,condition,campaigns,multi_chunk,max_chunks,max_tokens
0,0.1,300,8,2,256
1,0.3,300,8,2,256
2,0.6,300,9,2,256
3,clean,300,8,2,256


All actual MiniLM chunks are within the model limit.


In [34]:
# load official MiniLM
official_minilm={
    "clean":np.load(DATA_DIR/"missing_reference_embeddings_clean.npy"),
    .10:np.load(DATA_DIR/"missing_reference_embeddings_10pct.npy"),
    .30:np.load(DATA_DIR/"missing_reference_embeddings_30pct.npy"),
    .60:np.load(DATA_DIR/"missing_reference_embeddings_60pct.npy")
}

In [35]:
# exact recovery check
comparison=[]

for c in ["clean",.10,.30,.60]:
    A=reconstructed_minilm[c];B=official_minilm[c]
    comparison.append({
        "condition":c,
        "shape_match":A.shape==B.shape,
        "max_abs_diff":float(np.max(np.abs(A-B))),
        "mean_abs_diff":float(np.mean(np.abs(A-B))),
        "allclose":bool(np.allclose(A,B,atol=1e-6,rtol=1e-5))
    })

comparison_df=pd.DataFrame(comparison)
display(comparison_df)

assert comparison_df["shape_match"].all()
assert comparison_df["allclose"].all()

print("Exact Missing source conditions successfully recovered.")

,condition,shape_match,max_abs_diff,mean_abs_diff,allclose
0,clean,True,0.0,0.0,True
1,0.1,True,0.0,0.0,True
2,0.3,True,0.0,0.0,True
3,0.6,True,0.0,0.0,True


Exact Missing source conditions successfully recovered.


In [36]:
# save recovered plan
PLAN_DIR=DATA_DIR/"formal_defect_plans"
PLAN_DIR.mkdir(parents=True,exist_ok=True)

missing_plan.to_csv(PLAN_DIR/"missing_cross_channel_formal_plan_recovered.csv",index=False)

print("Recovered Missing plan saved.")

Recovered Missing plan saved.


In [37]:
# load frozen BGE from Entity
BGE_DIR=DATA_DIR/"robustness_bge"

with open(BGE_DIR/"entity_bge_configuration.json") as f:
    bge_config=json.load(f)

BGE_MODEL_NAME=bge_config["model"]
BGE_MODEL_REVISION=bge_config["revision"]
BGE_BATCH_SIZE=bge_config["batch_size"]

bge_model=SentenceTransformer(BGE_MODEL_NAME,revision=BGE_MODEL_REVISION)

print("Model:",BGE_MODEL_NAME)
print("Revision:",BGE_MODEL_REVISION)
print("Dimension:",bge_model.get_sentence_embedding_dimension())
print("Max sequence length:",bge_model.max_seq_length)

Model: BAAI/bge-base-en-v1.5
Revision: a5beb1e3e68b9ab74eb54cfd186867f64f240e1a
Dimension: 768
Max sequence length: 512


In [38]:
# confirm same campaign ordering
entity_bge_ids=pd.read_csv(BGE_DIR/"entity_bge_campaign_ids.csv")["campaign_id"].astype(str).to_numpy()
missing_clean_ids=missing_serialized["clean"]["campaign_id"].astype(str).to_numpy()

assert np.array_equal(entity_bge_ids,missing_clean_ids)

print("Same clean campaign IDs.")

Same clean campaign IDs.


In [39]:
# validate BGE chunking
chunk_records=[]

for c in ["clean",.10,.30,.60]:
    for _,row in missing_serialized[c].iterrows():
        chunks=split_campaign_into_chunks(row["serialized_text"],bge_model)
        lengths=[len(bge_model.tokenizer(x,add_special_tokens=True,truncation=False)["input_ids"]) for x in chunks]
        chunk_records.append({"condition":c,"campaign_id":row["campaign_id"],"n_chunks":len(chunks),"max_chunk_tokens":max(lengths)})

bge_chunk_validation=pd.DataFrame(chunk_records)

display(
    bge_chunk_validation.groupby("condition").agg(
        campaigns=("campaign_id","size"),
        multi_chunk=("n_chunks",lambda x:(x>1).sum()),
        max_chunks=("n_chunks","max"),
        max_chunk_tokens=("max_chunk_tokens","max")
    ).reset_index()
)

assert (bge_chunk_validation["max_chunk_tokens"]<=bge_model.max_seq_length).all()

print("Missing BGE chunk validation passed.")

,condition,campaigns,multi_chunk,max_chunks,max_chunk_tokens
0,0.1,300,0,1,261
1,0.3,300,0,1,261
2,0.6,300,0,1,261
3,clean,300,0,1,261


Missing BGE chunk validation passed.


In [40]:
# generate Missing BGE embeddings
missing_bge={}
missing_bge_runtime={}

for c in ["clean",.10,.30,.60]:
    X,runtime=encode_serialized_campaigns(missing_serialized[c],bge_model,batch_size=BGE_BATCH_SIZE)
    missing_bge[c]=X
    missing_bge_runtime[c]=runtime
    print(c,X.shape,f"{runtime:.2f}s")

Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

clean (300, 768) 205.23s
Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

0.1 (300, 768) 182.98s
Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

0.3 (300, 768) 184.81s
Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

0.6 (300, 768) 205.45s


In [41]:
# validate BGE matrices
for c,X in missing_bge.items():
    assert X.shape==(300,bge_model.get_sentence_embedding_dimension())
    assert np.isfinite(X).all()
    assert np.allclose(np.linalg.norm(X,axis=1),1.0,atol=1e-5)

print("All Missing BGE embeddings passed validation.")

All Missing BGE embeddings passed validation.


In [42]:
# check shared clean BGE
entity_bge_clean=np.load(BGE_DIR/"entity_bge_clean.npy")

max_clean_diff=np.max(np.abs(missing_bge["clean"]-entity_bge_clean))
print("Max clean BGE difference:",max_clean_diff)

assert np.allclose(missing_bge["clean"],entity_bge_clean,atol=1e-6,rtol=1e-5)

print("Entity and Missing share the same clean BGE representation.")

Max clean BGE difference: 0.0
Entity and Missing share the same clean BGE representation.


In [43]:
# verify changed ids
for s in SEVERITIES:
    distances=np.linalg.norm(missing_bge[s]-missing_bge["clean"],axis=1)
    changed_ids=set(missing_serialized["clean"].loc[distances>1e-8,"campaign_id"])
    expected_ids=set(missing_audits[s]["campaign_id"])

    print(f"{s:.0%} | changed={len(changed_ids)} | expected={len(expected_ids)} | exact match={changed_ids==expected_ids}")
    assert changed_ids==expected_ids

10% | changed=30 | expected=30 | exact match=True
30% | changed=90 | expected=90 | exact match=True
60% | changed=180 | expected=180 | exact match=True


In [44]:
# save artifacts
names={
    "clean":"missing_reference_bge_clean.npy",
    .10:"missing_reference_bge_10pct.npy",
    .30:"missing_reference_bge_30pct.npy",
    .60:"missing_reference_bge_60pct.npy"
}

for c,name in names.items():
    np.save(BGE_DIR/name,missing_bge[c])

missing_serialized["clean"][["campaign_id"]].to_csv(
    BGE_DIR/"missing_reference_bge_campaign_ids.csv",
    index=False
)

print("Missing BGE artifacts saved.")

Missing BGE artifacts saved.


In [45]:
# save config
missing_bge_config={
    "scenario":"Missing Cross-Channel Reference",
    "model":BGE_MODEL_NAME,
    "revision":BGE_MODEL_REVISION,
    "embedding_dimension":int(bge_model.get_sentence_embedding_dimension()),
    "normalized":True,
    "batch_size":BGE_BATCH_SIZE,
    "campaign_serialization":"same as formal MiniLM experiment",
    "chunking":"same message-boundary procedure as formal MiniLM experiment",
    "aggregation":"token-weighted chunk mean followed by L2 normalization",
    "source_conditions_recovered_exactly":True,
    "shared_clean_with_entity":True,
    "minilm_reconstruction_max_abs_diff":float(comparison_df["max_abs_diff"].max())
}

with open(BGE_DIR/"missing_reference_bge_configuration.json","w") as f:
    json.dump(missing_bge_config,f,indent=4)

print("Missing BGE configuration saved.")

Missing BGE configuration saved.


### Missing Cross-Channel Reference — Recovery and BGE Encoder Robustness Check

The formal Missing Cross-Channel Reference conditions were reconstructed from the original evaluation set using the same defect-generation procedure and fixed seed used in the previous experiment. The evaluation set contained **300 campaigns and 2,090 messages**, and the recovered defect plan contained **300 unique campaigns**. Severity was again defined through prevalence, producing **30, 90, and 180 affected campaigns** at the 10%, 30%, and 60% levels. The affected sets were nested, and the reconstructed campaign IDs matched the previously frozen affected-ID artifact exactly at all three severity levels. Only message text changed; all non-text fields were preserved, with exactly one modified message per affected campaign.

The reconstructed conditions were serialized using the same campaign-level pipeline as the original experiment. Each condition produced **300 serialized campaigns**. To verify that the original formal conditions had been recovered exactly, the serialized campaigns were re-encoded with the frozen `sentence-transformers/all-MiniLM-L6-v2` model, revision `1110a243fdf4706b3f48f1d95db1a4f5529b4d41`, producing 384-dimensional embeddings. Although the tokenizer reported that some complete serialized campaigns exceeded the 256-token model limit, the existing message-boundary chunking procedure handled these cases before encoding. The actual encoded chunks never exceeded 256 tokens; 8 clean, 8 at 10%, 8 at 30%, and 9 at 60% required two chunks. The reconstructed MiniLM embeddings were then compared with the previously saved formal embeddings. For clean, 10%, 30%, and 60%, the maximum and mean absolute differences were both **0.0**, with identical shapes and `allclose=True`. This confirms that the exact formal Missing Cross-Channel Reference conditions were successfully recovered.

For the secondary encoder robustness analysis, the same serialized campaign texts were encoded with the prespecified `BAAI/bge-base-en-v1.5` model, frozen at revision `a5beb1e3e68b9ab74eb54cfd186867f64f240e1a`. BGE produced **768-dimensional L2-normalized embeddings** and supported sequences up to 512 tokens. Under this encoder, all 300 campaigns in every condition fit into a single chunk; the maximum observed sequence length was 261 tokens. The same clean campaign ordering used for Entity Inconsistency was preserved.

The resulting BGE embedding matrices had shape **(300, 768)** for clean and all three severity levels and passed finite-value and unit-norm validation. The clean BGE matrix was exactly identical to the independently generated Entity Inconsistency clean BGE representation, with a maximum absolute difference of **0.0**, confirming that both robustness scenarios share the same clean representation. Finally, the BGE representation changed exactly the expected **30, 90, and 180 campaigns** at 10%, 30%, and 60% severity, with no additional campaigns affected. The recovered defect plan, BGE embeddings, campaign IDs, and encoder configuration were saved as frozen artifacts for the subsequent secondary robustness experiment.